In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModel
from sklearn.manifold import TSNE
import altair as alt
import base64
from io import BytesIO
import pickle
import matplotlib
import matplotlib.backends.backend_pdf
import pandas as pd
import numpy as np

In [ ]:
def batch_list_comprehension(data, batch_size):
    return [data[i:i + batch_size] for i in range(0, len(data), batch_size)]

In [ ]:
def image_formatter(im):
    height, width, _ = im.shape
    
    with BytesIO() as buffer:
        Image.fromarray(im).resize((width // 2, height // 2)).save(buffer, 'png')
        data = base64.encodebytes(buffer.getvalue()).decode('utf-8')
    
    return f"data:image/png;base64,{data}"

In [ ]:
all_meta = pd.read_csv("/nfs/mm-isilon/brainscans/dropbox/Slide_Incoming/metadata/master_spreadsheet/matched_su_mxa.csv")

In [ ]:
with open("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/np_slide_ingest/fsx/embs.pickle", "rb") as fd:
    embs = pickle.load(fd)
with open("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/np_slide_ingest/fsx/thumbnails.pickle", "rb") as fd:
    thumbnails = pickle.load(fd)
with open("/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/np_slide_ingest/fsx/path.pickle", "rb") as fd:
    path = pickle.load(fd)

## Save annotation

In [ ]:
tsne = TSNE(n_components=1, random_state=42)
embeddings_2d = tsne.fit_transform(np.stack(embs))
order = np.argsort(embeddings_2d.squeeze())

In [ ]:
thumb2 = [thumbnails[i] for i in order]
path2 = [path[i] for i in order]
title2 = []
for p in path2:
    curr_row = all_meta[all_meta["path"] == p]
    assert len(curr_row) == 1
    s = curr_row.iloc[0]
    title2.append(s["mxa_code"] + " / " + s['UM Accession']+" / "+s['Block']+" / "+s["Stain"])

In [ ]:
t2b = batch_list_comprehension(thumb2, 10)
l2b = batch_list_comprehension(title2, 10)

In [ ]:
pd.DataFrame([t.split(" / ") for t in title2], columns=["mxa", "su", "block", "stain"])#.to_csv("fsx.csv", index=False)

In [ ]:
#pdf = matplotlib.backends.backend_pdf.PdfPages("fsx.pdf")

for t2b_, l2b_ in tqdm(zip(t2b, l2b)):
    fig, axs = plt.subplots(10,1, figsize=(8.5*2,11*2), dpi=150)
    for ax in axs: ax.axis("off")
        
    for t,l,ax in zip(t2b_, l2b_, axs):
        ax.imshow(t)
        ax.set_title(l)
    #pdf.savefig(fig)
    plt.close(fig)
#pdf.close()

# This is where you actually do the annotation

Come back once you are done

## Visualize annotation

In [ ]:
annotation = pd.read_csv("fsx_annot.csv")

In [ ]:
annotation

In [ ]:
mxa_with_annot = all_meta[all_meta["Block"].str.contains("FS") & (all_meta["Stain"]=="H&E")].merge(annotation, left_on="mxa_code", right_on="mxa", how="left")[["mxa_code", "path", "x_p"]]


In [ ]:
order_map = {val: index for index, val in enumerate(path)}
mxa_with_annot['sort_order'] = mxa_with_annot['path'].map(order_map)
mxa_with_annot = mxa_with_annot.sort_values(by='sort_order').drop('sort_order', axis=1)


In [ ]:
tsne = TSNE(n_components=2, random_state=42)
embeddings_2d = tsne.fit_transform(np.stack(embs))
mxa_with_annot["x"] = embeddings_2d[:, 0]
mxa_with_annot["y"] = embeddings_2d[:, 1]

In [ ]:
mxa_with_annot["image"] = [image_formatter(i) for i in thumbnails]

In [ ]:
# Create an Altair scatter plot
scatter_plot = alt.Chart(mxa_with_annot).mark_circle(size=60).encode(
    x='x',
    y='y',
    color="x_p",
    tooltip=['image']
)

scatter_plot.properties(
    width=800,
    height=800
).interactive()